# README to Label Studio Converter

This notebook converts `README.md` files from the `github_metadata_dataset` directory into a JSON format suitable for Label Studio. It processes all README files per project (first-level subdirectory), deduplicates identical contents, merges unique sentences, and outputs one `README.json` per project. The goal is to prepare content for annotating Codemeta terms.

Features:
- Groups READMEs by project directory.
- Deduplicates identical README contents using MD5 hashing.
- Merges unique sentences from all READMEs in a project into one `README.json`.
- Improved sentence splitting to handle abbreviations, initials, and citations.
- Filters out very short sentences (< 10 characters) except in citations.
- Preserves citation entries as single tasks in the 'Cite' section.
- Outputs in Label Studio format with `text`, `section`, `repo`, and `source_file` fields.

## 1. Import Libraries and Setup

Import required libraries for Markdown parsing, text processing, file handling, and hashing.

In [82]:
# Install necessary libraries if not already installed
!pip install markdown-it-py mdit-py-plugins tqdm ipywidgets --quiet
# Import libraries
from markdown_it import MarkdownIt
from mdit_py_plugins.footnote import footnote_plugin
import json
import os
import glob
import re
import html
from tqdm.notebook import tqdm
import hashlib


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


## 2. Define Cleaning Function

Cleans Markdown text to normalize it for annotation, removing unnecessary elements (e.g., images, badges, code blocks) while preserving content.

In [83]:
def clean_readme_text(text: str, software_name: str = None) -> str:
    """Clean README text while preserving structure and readability.
    
    Args:
        text: README text to clean
        software_name: Name of the software to mask in installation commands
    """
    if not text or not text.strip():
        return ""
    
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    
    # Remove images but keep badges
    text = re.sub(r'!\[[^\]]*\]\([^)]+\)', '', text)
    
    # Clean markdown links - extract text and URL
    text = re.sub(r'\[([^\]]+)\]\(([^)]+)\)', r'[\1] \2', text)
    
    # Remove code fence markers but keep the content
    text = re.sub(r'^```\s*\w*\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^~~~\s*\w*\s*$', '', text, flags=re.MULTILINE)
    
    # Remove inline code backticks but keep content
    text = re.sub(r'`([^`]+)`', r'\1', text)
    
    # Mask installation commands that mention the software name
    if software_name:
        # Case-insensitive matching for pip install, conda install, etc.
        install_patterns = [
            rf'\bpip\s+install\s+{re.escape(software_name)}\b',
            rf'\bconda\s+install\s+{re.escape(software_name)}\b',
            rf'\bnpm\s+install\s+{re.escape(software_name)}\b',
            rf'\byarn\s+add\s+{re.escape(software_name)}\b',
            rf'\bgem\s+install\s+{re.escape(software_name)}\b',
            rf'\binstall\.packages\s*\(\s*["\']?{re.escape(software_name)}["\']?\s*\)',  # R
        ]
        
        for pattern in install_patterns:
            text = re.sub(pattern, 'XXX_SOFTWARE_INSTALLATION', text, flags=re.IGNORECASE)
    
    # Clean markdown formatting
    text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)
    text = re.sub(r'\*([^*]+)\*', r'\1', text)
    text = re.sub(r'^[ \t]*[=\-_*]{3,}[ \t]*$', '', text, flags=re.MULTILINE)
    
    # Protect list items: add double newline after each list item to preserve them
    text = re.sub(r'^[ \t]*[-*+][ \t]+(.+)$', r'\1\n', text, flags=re.MULTILINE)
    text = re.sub(r'^[ \t]*\d+\.[ \t]+(.+)$', r'\1\n', text, flags=re.MULTILINE)
    
    # Clean blockquotes
    text = re.sub(r'^[ \t]*>+[ \t]*', '', text, flags=re.MULTILINE)
    
    # Normalize excessive whitespace
    text = re.sub(r'[ \t]+', ' ', text)  # Multiple spaces/tabs to single space
    
    # Protect headers: add an extra newline after headers so they don't get merged
    text = re.sub(r'(^#{1,6}\s+.+)$', r'\1\n', text, flags=re.MULTILINE)
    
    # Replace single newlines with spaces (merge lines into paragraphs)
    # But keep intentional paragraph breaks (double newlines), headers, and list items
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)
    
    # Normalize multiple newlines to double newlines (paragraph breaks)
    text = re.sub(r'\n{2,}', '\n\n', text)
    
    # Clean up extra spaces around newlines
    text = re.sub(r' *\n *', '\n', text)
    
    # Remove empty lines that only contain whitespace
    text = re.sub(r'\n\s*\n', '\n\n', text)
    
    return text.strip()

## 3. Define File Reading Function

Safely reads files with various encodings.

In [84]:
def safe_read_file(file_path):
    """
    Attempts to read a file with various encodings.
    
    Args:
        file_path (str): Path to the file
        
    Returns:
        str: File content or empty string on error
    """
    encodings = ['utf-8', 'latin-1', 'windows-1252', 'ascii']
    for encoding in encodings:
        try:
            with open(file_path, 'r', encoding=encoding) as f:
                return f.read()
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"Error reading {file_path}: {str(e)}")
            return ""
    print(f"Could not read {file_path} with any encoding.")
    return ""

## 4. Parse README to Sections

Parses Markdown text into structured sections based on headings.

In [ ]:
def parse_readme_to_sections(md_text):
    """
    Parses Markdown text intelligently:
    1. If entire text fits in token limit, return as single section
    2. If too large, split by headers and merge sections to create balanced chunks
    3. Only split individual sections if they exceed the limit on their own
    
    Args:
        md_text (str): Markdown text
        
    Returns:
        list: List of sections with title and content
    """
    MAX_TOKENS = 480 
    
    if not md_text.strip():
        return []
    
    # Check if entire text fits within limit
    estimated_tokens = len(md_text) / 4  # 1 token ≈ 4 chars
    
    if estimated_tokens <= MAX_TOKENS:
        # Return entire README as single section
        return [{
            "section": "README",
            "content": md_text.strip()
        }]
    
    # Text is too large, split by headers first
    sections = _split_by_headers(md_text)
    
    # Merge sections together to create balanced chunks
    merged_chunks = _merge_sections_into_chunks(sections, MAX_TOKENS)
    
    return merged_chunks


def _split_by_headers(md_text):
    """
    Splits markdown text by top-level headers (# only, not ## or ###).
    Preserves all newlines and spacing in content.
    
    Args:
        md_text (str): Markdown text
        
    Returns:
        list: List of sections with title and content
    """
    import re
    
    sections = []
    current_section = "Introduction"
    current_content = []
    found_any_header = False
    
    lines = md_text.split('\n')
    
    for line in lines:
        # Check if line is a TOP-LEVEL header (single # only)
        header_match = re.match(r'^#{1}\s+(.+)$', line)
        if header_match:
            found_any_header = True
            # Save previous section (only if it has content beyond the title)
            if current_content:
                content = '\n'.join(current_content).strip()
                if content and content != current_section:  # Don't save if only title exists
                    sections.append({
                        "section": current_section,
                        "content": content
                    })
            # Start new section with header as title
            current_section = header_match.group(1).strip()
            # Include section title in content (without the # symbol)
            current_content = [current_section]
        else:
            current_content.append(line)
    
    # Save last section (only if it has content)
    if current_content:
        content = '\n'.join(current_content).strip()
        if content:  # Only add if there's actual content
            sections.append({
                "section": current_section,
                "content": content
            })
    
    # If no sections with content found, return entire text as one section
    if not sections:
        sections.append({
            "section": "Content",
            "content": md_text.strip()
        })
    
    return sections


def _merge_sections_into_chunks(sections, max_tokens):
    """
    Merges multiple sections together into balanced chunks.
    Tries to fit as many sections as possible into each chunk without exceeding limit.
    
    Args:
        sections (list): List of section dictionaries
        max_tokens (int): Maximum tokens per chunk
        
    Returns:
        list: List of merged chunks
    """
    max_chars = max_tokens * 4  # Conservative estimate: 1 token ≈ 4 chars
    chunks = []
    
    current_chunk_sections = []
    current_chunk_length = 0
    
    for section in sections:
        section_length = len(section["content"])
        
        # If this single section exceeds the limit, handle it separately
        if section_length > max_chars:
            # Save current chunk if it has content
            if current_chunk_sections:
                chunks.append(_create_merged_chunk(current_chunk_sections))
                current_chunk_sections = []
                current_chunk_length = 0
            
            # Split the large section into multiple chunks
            split_chunks = _split_section_into_chunks(
                section["content"],
                section["section"],
                max_tokens
            )
            chunks.extend(split_chunks)
        
        # If adding this section would exceed limit, save current chunk and start new one
        elif current_chunk_length + section_length > max_chars and current_chunk_sections:
            chunks.append(_create_merged_chunk(current_chunk_sections))
            current_chunk_sections = [section]
            current_chunk_length = section_length
        
        # Add section to current chunk
        else:
            current_chunk_sections.append(section)
            current_chunk_length += section_length
    
    # Save last chunk
    if current_chunk_sections:
        chunks.append(_create_merged_chunk(current_chunk_sections))
    
    return chunks


def _create_merged_chunk(sections):
    """
    Creates a single chunk from multiple sections.
    
    Args:
        sections (list): List of section dictionaries to merge
        
    Returns:
        dict: Merged chunk with combined section names and content
    """
    if len(sections) == 1:
        return sections[0]
    
    # Combine section names
    section_names = [s["section"] for s in sections]
    combined_name = " + ".join(section_names[:3])  # Limit to first 3 names
    if len(section_names) > 3:
        combined_name += f" (and {len(section_names) - 3} more)"
    
    # Combine content with section separators
    combined_content = "\n\n".join([s["content"] for s in sections])
    
    return {
        "section": combined_name,
        "content": combined_content
    }

def _split_section_into_chunks(content, section_name, max_tokens):
    """
    Splits a large section into smaller chunks that fit within token limit.
    Preserves complete subsections - never splits in the middle of a subsection.
    
    Args:
        content (str): Section content to split
        section_name (str): Name of the section
        max_tokens (int): Maximum tokens per chunk
        
    Returns:
        list: List of section chunks
    """
    import re
    
    max_chars = max_tokens * 4  # Conservative estimate: 1 token ≈ 4 chars
    chunks = []
    chunk_index = 1
    
    # First, try to identify subsections (## ### etc.)
    subsections = []
    current_subsection = {"header": None, "content": []}
    
    lines = content.split('\n')
    
    for line in lines:
        # Check if this is a subsection header (##, ###, etc.)
        subsection_match = re.match(r'^(#{2,6})\s+(.+)$', line)
        if subsection_match:
            # Save previous subsection if it has content
            if current_subsection["content"] or current_subsection["header"]:
                subsections.append(current_subsection)
            # Start new subsection
            current_subsection = {
                "header": line,
                "content": []
            }
        else:
            current_subsection["content"].append(line)
    
    # Save last subsection
    if current_subsection["content"] or current_subsection["header"]:
        subsections.append(current_subsection)
    
    # If no subsections found, treat entire content as one subsection
    if not subsections or (len(subsections) == 1 and not subsections[0]["header"]):
        # No subsections, fall back to paragraph-based splitting
        return _split_by_paragraphs(content, section_name, max_tokens)
    
    # Now group subsections into chunks, keeping complete subsections together
    current_chunk = []
    current_length = 0
    
    for subsection in subsections:
        # Build subsection text
        subsection_parts = []
        if subsection["header"]:
            subsection_parts.append(subsection["header"])
        subsection_parts.extend(subsection["content"])
        subsection_text = '\n'.join(subsection_parts)
        subsection_length = len(subsection_text)
        
        # If single subsection exceeds limit, we need to split it
        if subsection_length > max_chars:
            # Save current chunk if it has content
            if current_chunk:
                chunk_content = '\n'.join(current_chunk)
                chunks.append({
                    "section": f"{section_name} (part {chunk_index})",
                    "content": chunk_content
                })
                chunk_index += 1
                current_chunk = []
                current_length = 0
            
            # Split this large subsection by paragraphs
            subsection_chunks = _split_by_paragraphs(
                subsection_text,
                section_name,
                max_tokens
            )
            
            # Add subsection chunks with proper indexing
            for sub_chunk in subsection_chunks:
                chunks.append({
                    "section": f"{section_name} (part {chunk_index})",
                    "content": sub_chunk["content"]
                })
                chunk_index += 1
        
        # If adding this subsection would exceed limit, save current chunk
        elif current_length + subsection_length + 1 > max_chars and current_chunk:
            chunk_content = '\n'.join(current_chunk)
            chunks.append({
                "section": f"{section_name} (part {chunk_index})",
                "content": chunk_content
            })
            chunk_index += 1
            current_chunk = [subsection_text]
            current_length = subsection_length
        
        # Add subsection to current chunk
        else:
            current_chunk.append(subsection_text)
            current_length += subsection_length + 1  # +1 for newline
    
    # Save last chunk
    if current_chunk:
        chunk_content = '\n'.join(current_chunk)
        section_label = f"{section_name} (part {chunk_index})" if chunk_index > 1 else section_name
        chunks.append({
            "section": section_label,
            "content": chunk_content
        })
    
    return chunks


def _split_by_paragraphs(content, section_name, max_tokens):
    """
    Fallback method to split content by paragraphs when no subsections exist.
    
    Args:
        content (str): Content to split
        section_name (str): Name of the section
        max_tokens (int): Maximum tokens per chunk
        
    Returns:
        list: List of section chunks
    """
    import re
    
    max_chars = max_tokens * 4
    chunks = []
    chunk_index = 1
    
    # Split by double newlines (paragraphs)
    paragraphs = content.split('\n\n')
    
    current_chunk = []
    current_length = 0
    
    for para in paragraphs:
        para_length = len(para)
        
        # If single paragraph is too large, split by lines
        if para_length > max_chars:
            # Save current chunk
            if current_chunk:
                chunk_content = '\n\n'.join(current_chunk)
                chunks.append({
                    "section": f"{section_name} (part {chunk_index})",
                    "content": chunk_content
                })
                chunk_index += 1
                current_chunk = []
                current_length = 0
            
            # Split paragraph by lines
            lines = para.split('\n')
            temp_lines = []
            temp_length = 0
            
            for line in lines:
                line_length = len(line)
                
                if line_length > max_chars:
                    # Save accumulated lines
                    if temp_lines:
                        chunks.append({
                            "section": f"{section_name} (part {chunk_index})",
                            "content": '\n'.join(temp_lines)
                        })
                        chunk_index += 1
                        temp_lines = []
                        temp_length = 0
                    
                    # Split very long line by sentences
                    sentences = re.split(r'(?<=[.!?])\s+', line)
                    for sentence in sentences:
                        if len(sentence) > max_chars:
                            # Character-level split as last resort
                            for i in range(0, len(sentence), max_chars):
                                chunks.append({
                                    "section": f"{section_name} (part {chunk_index})",
                                    "content": sentence[i:i + max_chars]
                                })
                                chunk_index += 1
                        else:
                            chunks.append({
                                "section": f"{section_name} (part {chunk_index})",
                                "content": sentence
                            })
                            chunk_index += 1
                
                elif temp_length + line_length + 1 > max_chars:
                    if temp_lines:
                        chunks.append({
                            "section": f"{section_name} (part {chunk_index})",
                            "content": '\n'.join(temp_lines)
                        })
                        chunk_index += 1
                    temp_lines = [line]
                    temp_length = line_length
                else:
                    temp_lines.append(line)
                    temp_length += line_length + 1
            
            # Save remaining lines
            if temp_lines:
                chunks.append({
                    "section": f"{section_name} (part {chunk_index})",
                    "content": '\n'.join(temp_lines)
                })
                chunk_index += 1
        
        # If adding paragraph exceeds limit, save current chunk
        elif current_length + para_length + 2 > max_chars:
            if current_chunk:
                chunk_content = '\n\n'.join(current_chunk)
                chunks.append({
                    "section": f"{section_name} (part {chunk_index})",
                    "content": chunk_content
                })
                chunk_index += 1
            current_chunk = [para]
            current_length = para_length
        
        # Add paragraph to current chunk
        else:
            current_chunk.append(para)
            current_length += para_length + 2
    
    # Save last chunk
    if current_chunk:
        chunk_content = '\n\n'.join(current_chunk)
        section_label = f"{section_name} (part {chunk_index})" if chunk_index > 1 else section_name
        chunks.append({
            "section": section_label,
            "content": chunk_content
        })
    
    return chunks

## 6. Define Hash Function

Utility function for computing MD5 hashes for deduplication.

In [86]:
def compute_hash(text_or_bytes):
    """
    Computes MD5 hash for deduplication.
    
    Args:
        text_or_bytes: String or bytes to hash
        
    Returns:
        str: MD5 hash
    """
    if isinstance(text_or_bytes, str):
        return hashlib.md5(text_or_bytes.encode('utf-8', errors='ignore')).hexdigest()
    return hashlib.md5(text_or_bytes).hexdigest()

## 7. Batch Process All README Files

Processes all `README.md` files per project, deduplicates identical contents, merges unique sentences, and outputs one `README.json` per project in Label Studio format.

In [87]:
def get_projects(base_dir):
    """
    Gets list of project directories (first-level subdirs).
    
    Args:
        base_dir (str): Base directory
        
    Returns:
        list: List of project directory names
    """
    try:
        return [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    except Exception as e:
        print(f"Error listing directories in {base_dir}: {str(e)}")
        return []

def search_readme_files(project_path):
    """
    Searches for all README file variants in a project directory.
    
    Args:
        project_path (str): Path to the project directory
        
    Returns:
        list: List of paths to README files found
    """
    # Define README patterns to search for
    readme_patterns = [
        "readme.md", "readme.rst", "readme.dev.md", "readme",
        "readme.rmd", "readme.qmd", "readme.txt", "readme-dev.md",
        "readme.dev.rst", "readme.compiling", "readme.legal",
        "readme.git", "readme.jsoncpp", "readme.python",
        "readme.linux", "readme.sunos", "readme.win32",
        "readme_*.md", "readme-*.md", "readme_*.rst",
        "readme_*.rtf", "readme*.md"
    ]
    
    readme_files = []
    
    try:
        # Search for all README variants
        for pattern in readme_patterns:
            pattern_path = os.path.join(project_path, "**", pattern)
            found_files = glob.glob(pattern_path, recursive=True)
            readme_files.extend(found_files)
        
        # Remove duplicates (in case patterns overlap)
        readme_files = list(set(readme_files))
        
    except Exception as e:
        print(f"Error finding READMEs in {project_path}: {str(e)}")
        return []
    
    return readme_files


def process_all_readmes(base_dir, output_dir):
    """
    Processes READMEs per project, merging unique contents into one JSON.

    Args:
        base_dir (str): Directory containing project subdirectories
        output_dir (str): Directory for JSON output files

    Returns:
        dict: Statistics on processing
    """
    projects = get_projects(base_dir)
    print(f"Found {len(projects)} projects.")

    os.makedirs(output_dir, exist_ok=True)

    stats = {
        'total_projects': len(projects),
        'processed_projects': 0,
        'total_readmes_found': 0,
        'skipped_duplicate_readmes': 0,
        'unique_readmes_total': 0,
        'total_unique_sentences': 0,
        'failed_projects': 0,
        'empty_projects': 0
    }

    for project in tqdm(projects, desc="Processing projects"):
        project_path = os.path.join(base_dir, project)
        readme_files = search_readme_files(project_path)

        stats['total_readmes_found'] += len(readme_files)
        if not readme_files:
            stats['empty_projects'] += 1
            continue

        # Deduplicate README contents
        content_hashes = set()
        unique_readmes = []
        for readme_file in readme_files:
            md_text = safe_read_file(readme_file)
            if not md_text.strip():
                continue
            content_hash = compute_hash(md_text)
            if content_hash not in content_hashes:
                content_hashes.add(content_hash)
                unique_readmes.append((readme_file, md_text))
            else:
                stats['skipped_duplicate_readmes'] += 1

        num_unique = len(unique_readmes)
        stats['unique_readmes_total'] += num_unique
        if num_unique == 0:
            stats['empty_projects'] += 1
            continue

        # Merge sentences from unique READMEs
        seen_sentence_hashes = set()
        all_project_sentences = []

        for readme_file, md_text in unique_readmes:
            try:
                sections = parse_readme_to_sections(md_text)
                for section in sections:
                    # Clean the content
                    cleaned_content = clean_readme_text(section['content'])
                    content_hash = compute_hash(cleaned_content)
                    
                    if content_hash not in seen_sentence_hashes:
                        seen_sentence_hashes.add(content_hash)
                        all_project_sentences.append({
                            'text': cleaned_content,
                            'section': section['section'],
                            'repo': project,
                            'source_file': os.path.relpath(readme_file, project_path)
                        })
            except Exception as e:
                print(f"Error processing {readme_file}: {str(e)}")
                continue

        # Output one JSON per project
        project_out_dir = os.path.join(output_dir, project)
        out_file = os.path.join(project_out_dir, "README.json")
        try:
            os.makedirs(project_out_dir, exist_ok=True)
            tasks = [{"data": sent} for sent in all_project_sentences]
            with open(out_file, 'w', encoding='utf-8') as f:
                json.dump(tasks, f, indent=2)

            num_sentences = len(all_project_sentences)
            stats['processed_projects'] += 1
            stats['total_unique_sentences'] += num_sentences
            print(f"Project '{project}': Merged {num_unique} unique READMEs into {num_sentences} sentences -> {out_file}")
        except Exception as e:
            print(f"Error writing JSON for project {project}: {str(e)}")
            stats['failed_projects'] += 1

    return stats


## 8. Check Directory Structure

Verify the directory structure.

In [88]:
base_dir = "../data/raw"
if not os.path.exists(base_dir):
    print(f"ERROR: Directory {base_dir} does not exist.")
else:
    print(f"Directory {base_dir} exists.")
    projects = get_projects(base_dir)
    print(f"Found {len(projects)} projects.")
    print("Sample projects:")
    for project in projects[:5]:
        print(f"- {project}")
        readme_path = os.path.join(base_dir, project, "README.md")
        print(f"  README.md exists" if os.path.exists(readme_path) else f"  README.md does not exist")

Directory ../data/raw exists.
Found 497 projects.
Sample projects:
- 224_maftools
  README.md exists
- 250_COPASI
  README.md exists
- 130_dCacheFS
  README.md does not exist
- 290_argopy
  README.md exists
- 028_CryoGrid
  README.md exists


## 9. Process All README Files

Process all files and display statistics.

In [89]:
output_dir = "../data/labelstudio_json_data"
stats = process_all_readmes(base_dir, output_dir)

print("\nProcessing Statistics:")
print(f"Total projects: {stats['total_projects']}")
print(f"Processed projects: {stats['processed_projects']}")
print(f"Failed projects: {stats['failed_projects']}")
print(f"Empty projects (no valid READMEs): {stats['empty_projects']}")
print(f"Total READMEs found: {stats['total_readmes_found']}")
print(f"Skipped duplicate READMEs: {stats['skipped_duplicate_readmes']}")
print(f"Total unique READMEs processed: {stats['unique_readmes_total']}")
print(f"Total unique sentences: {stats['total_unique_sentences']}")
if stats['processed_projects'] > 0:
    print(f"Average unique sentences per project: {stats['total_unique_sentences'] / stats['processed_projects']:.2f}")

Found 497 projects.


Processing projects:   0%|          | 0/497 [00:00<?, ?it/s]

Project '224_maftools': Merged 1 unique READMEs into 14 sentences -> ../data/labelstudio_json_data/224_maftools/README.json
Project '250_COPASI': Merged 4 unique READMEs into 8 sentences -> ../data/labelstudio_json_data/250_COPASI/README.json
Project '130_dCacheFS': Merged 1 unique READMEs into 3 sentences -> ../data/labelstudio_json_data/130_dCacheFS/README.json
Project '290_argopy': Merged 1 unique READMEs into 8 sentences -> ../data/labelstudio_json_data/290_argopy/README.json
Project '028_CryoGrid': Merged 1 unique READMEs into 1 sentences -> ../data/labelstudio_json_data/028_CryoGrid/README.json
Project '270_dtscalibration': Merged 1 unique READMEs into 4 sentences -> ../data/labelstudio_json_data/270_dtscalibration/README.json
Project '368_UrbEm_-_Urban_Emission_downscaling_for_air_quality_modeling': Merged 1 unique READMEs into 4 sentences -> ../data/labelstudio_json_data/368_UrbEm_-_Urban_Emission_downscaling_for_air_quality_modeling/README.json
Project '053_Eventdisplay': Merg

## 10. Analyze Generated Data

Analyze the generated JSON files and create statistics.

In [90]:
def analyze_json_files(output_dir):
    """
    Analyzes generated JSON files for statistics.
    
    Args:
        output_dir (str): Directory containing JSON files
        
    Returns:
        dict: Analysis statistics
    """
    json_files = glob.glob(os.path.join(output_dir, "**", "*.json"), recursive=True)
    analysis = {
        'total_files': len(json_files),
        'total_sentences': 0,
        'section_counts': {},
        'sentence_lengths': [],
        'repo_sentence_counts': {},
        'source_file_counts': {}
    }
    
    for json_file in json_files:
        repo_sentences = 0
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                for item in data:
                    sentence_data = item.get('data', {})
                    analysis['total_sentences'] += 1
                    repo_sentences += 1
                    section = sentence_data.get('section', 'Unknown')
                    analysis['section_counts'][section] = analysis['section_counts'].get(section, 0) + 1
                    analysis['sentence_lengths'].append(len(sentence_data.get('text', '')))
                    source_file = sentence_data.get('source_file', 'Unknown')
                    analysis['source_file_counts'][source_file] = analysis['source_file_counts'].get(source_file, 0) + 1
            repo_name = os.path.basename(os.path.dirname(json_file))
            analysis['repo_sentence_counts'][repo_name] = repo_sentences
        except Exception as e:
            print(f"Error analyzing {json_file}: {str(e)}")
    
    analysis['avg_sentence_length'] = sum(analysis['sentence_lengths']) / len(analysis['sentence_lengths']) if analysis['sentence_lengths'] else 0
    analysis['top_sections'] = sorted(analysis['section_counts'].items(), key=lambda x: x[1], reverse=True)[:10]
    analysis['top_repos'] = sorted(analysis['repo_sentence_counts'].items(), key=lambda x: x[1], reverse=True)[:10]
    analysis['top_source_files'] = sorted(analysis['source_file_counts'].items(), key=lambda x: x[1], reverse=True)[:10]
    
    return analysis

if os.path.exists(output_dir):
    analysis = analyze_json_files(output_dir)
    print("\nData Analysis:")
    print(f"Processed JSON files: {analysis['total_files']}")
    print(f"Total sentences: {analysis['total_sentences']}")
    print(f"Average sentence length: {analysis['avg_sentence_length']:.2f} characters")
    print("\nTop 10 Sections:")
    for section, count in analysis['top_sections']:
        print(f"- {section}: {count} sentences")
    print("\nTop 10 Repos by Sentence Count:")
    for repo, count in analysis['top_repos']:
        print(f"- {repo}: {count} sentences")
    print("\nTop 10 Source Files by Sentence Count:")
    for source_file, count in analysis['top_source_files']:
        print(f"- {source_file}: {count} sentences")
else:
    print(f"Output directory {output_dir} does not exist yet.")


Data Analysis:
Processed JSON files: 494
Total sentences: 2713
Average sentence length: 1052.56 characters

Top 10 Sections:
- Introduction (part 1): 131 sentences
- Introduction (part 2): 131 sentences
- Introduction (part 3): 89 sentences
- Introduction: 88 sentences
- README: 80 sentences
- Introduction (part 4): 60 sentences
- Introduction (part 5): 41 sentences
- Introduction (part 6): 32 sentences
- Introduction (part 7): 28 sentences
- Introduction (part 8): 21 sentences

Top 10 Repos by Sentence Count:
- 168_CoupledNODE: 55 sentences
- 496_Rankings_Reloaded: 39 sentences
- 222_GridFormat: 36 sentences
- 265_DIANNA: 31 sentences
- 212_Parcels: 23 sentences
- 373_hyphe: 23 sentences
- 247_cwltool: 23 sentences
- 165_SMG2S: 22 sentences
- 328_Inseq: 22 sentences
- 210_asreview-simulation: 21 sentences

Top 10 Source Files by Sentence Count:
- readme.md: 2403 sentences
- readme.rst: 162 sentences
- readme.dev.md: 80 sentences
- readme.rmd: 33 sentences
- readme: 14 sentences
- rea